In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
from pathlib import Path

import numpy as np
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_fasta_snapshot as ncbi_fasta_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.sweep_genes_snapshot as sweep_genes_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_fasta_snapshot_module = importlib.reload(ncbi_fasta_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
sweep_genes_snapshot_module = importlib.reload(sweep_genes_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_ncbi_protein_fasta_snapshot = (
    ncbi_fasta_snapshot_module.resolve_ncbi_protein_fasta_snapshot
)
resolve_sweep_genes_snapshot = sweep_genes_snapshot_module.resolve_sweep_genes_snapshot

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\pago-proj\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Define hidden local SWeeP source
# =============================================================================

SCRIPTS_SWEEP_ROOT_DIRECTORY = PROJECT_ROOT / "scripts" / "sweep-2.1.3.0"

if not (SCRIPTS_SWEEP_ROOT_DIRECTORY / "sweep").exists():
    raise FileNotFoundError(
        "The hidden local SWeeP source directory was not found at "
        f"{SCRIPTS_SWEEP_ROOT_DIRECTORY}."
    )

print(f"Local SWeeP directory: {SCRIPTS_SWEEP_ROOT_DIRECTORY}")
print("Notebook execution will call the SWeeP Genes snapshot module.")

FileNotFoundError: The hidden local SWeeP source directory was not found at C:\pago-proj\pAgo-project\scripts\sweep-2.1.3.0.

In [4]:
# =============================================================================
# CELL 4 — Define FASTA and SWeeP Genes configuration
# =============================================================================

XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)
FASTA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_fasta"
)
SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "03-features" / "sweep_genes"
)

FASTA_SNAPSHOT_MODE = SnapshotMode.reuse_latest
SWEEP_GENES_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
SWEEP_GENES_MASKS = [
    [1, 1, 0, 0, 1],
    [1, 1, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [1, 1, 0, 1, 0],
]
PROJECTED_DIMENSIONS_PER_MASK = 700
TOTAL_EXPECTED_EMBEDDING_DIMENSIONS = (
    len(SWEEP_GENES_MASKS) * PROJECTED_DIMENSIONS_PER_MASK
)
SWEEP_GENES_EMBEDDINGS_FILE_NAME = (
    f"sweep_genes_embeddings_{TOTAL_EXPECTED_EMBEDDING_DIMENSIONS}D.npy"
)
COMPOSITION = "binary"
PROJECTION = True
RANDOM_SEED = 42
CHUNK_SIZE = 256
AVAILABLE_CPUS = os.cpu_count() or 1
N_JOBS = (
    1
    if SWEEP_GENES_SNAPSHOT_MODE == SnapshotMode.reuse_latest
    else min(8, max(1, AVAILABLE_CPUS - 1))
)
UPDATE_LATEST_DIRECTORY = True

if TOTAL_EXPECTED_EMBEDDING_DIMENSIONS != 2800:
    raise RuntimeError(
        "The configured masks must produce 2800 dimensions in total. "
        f"Got {TOTAL_EXPECTED_EMBEDDING_DIMENSIONS}."
    )

print(f"FASTA snapshot root directory: {FASTA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"SWeeP Genes output root directory: {SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY}")
print(
    "SWeeP matrix latest output path: "
    f"{SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY / 'latest' / SWEEP_GENES_EMBEDDINGS_FILE_NAME}"
)
print(f"SWeeP Genes snapshot mode: {SWEEP_GENES_SNAPSHOT_MODE}")
print(f"Masks: {SWEEP_GENES_MASKS}")
print(f"Projected dimensions per mask: {PROJECTED_DIMENSIONS_PER_MASK}")
print(f"Total expected embedding dimensions: {TOTAL_EXPECTED_EMBEDDING_DIMENSIONS}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"n_jobs: {N_JOBS}")

FASTA snapshot root directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta
SWeeP Genes output root directory: C:\pago-proj\pAgo-project\data\03-features\sweep_genes
SWeeP matrix latest output path: C:\pago-proj\pAgo-project\data\03-features\sweep_genes\latest\sweep_genes_embeddings_2800D.npy
SWeeP Genes snapshot mode: reuse_latest_or_create
Masks: [[1, 1, 0, 0, 1], [1, 1, 1, 0, 0], [1, 0, 1, 0, 1], [1, 1, 0, 1, 0]]
Projected dimensions per mask: 700
Total expected embedding dimensions: 2800
Chunk size: 256
n_jobs: 8


In [5]:
# =============================================================================
# CELL 5 — Resolve active FASTA snapshot
# =============================================================================

fasta_snapshot_payload = resolve_ncbi_protein_fasta_snapshot(
    snapshot_mode=FASTA_SNAPSHOT_MODE,
    snapshot_root_directory=FASTA_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

protein_fasta_snapshot_directory = fasta_snapshot_payload["snapshot_directory"]
protein_fasta_manifest_file_path = fasta_snapshot_payload["manifest_file_path"]
protein_fasta_file_path = fasta_snapshot_payload["fasta_file_path"]
protein_fasta_manifest_payload = fasta_snapshot_payload["manifest"]
protein_fasta_output_directory = protein_fasta_snapshot_directory

protein_fasta_file_sha256 = sha256_of_file(input_file_path=protein_fasta_file_path)
protein_fasta_manifest_file_sha256 = sha256_of_file(
    input_file_path=protein_fasta_manifest_file_path,
)

source_metadata_snapshot_relative_path = protein_fasta_manifest_payload[
    "source_metadata_snapshot_relative_path"
]
source_xml_snapshot_relative_path = protein_fasta_manifest_payload[
    "source_xml_snapshot_relative_path"
]

source_metadata_snapshot_directory = (
    METADATA_SNAPSHOT_ROOT_DIRECTORY / source_metadata_snapshot_relative_path
)
source_xml_snapshot_directory = (
    XML_SNAPSHOT_ROOT_DIRECTORY / source_xml_snapshot_relative_path
)

print("Resolved FASTA snapshot successfully.")
print(f"FASTA snapshot directory: {protein_fasta_snapshot_directory}")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"FASTA manifest file path: {protein_fasta_manifest_file_path}")

Resolved FASTA snapshot successfully.
FASTA snapshot directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest
FASTA file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\protein_sequences.fasta
FASTA manifest file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\manifest.json


In [6]:
# =============================================================================
# CELL 6 — Bind SWeeP Genes module entry points
# =============================================================================

load_latest_sweep_genes_snapshot = (
    sweep_genes_snapshot_module.load_latest_sweep_genes_snapshot
)
latest_sweep_genes_snapshot_is_available = (
    sweep_genes_snapshot_module.latest_sweep_genes_snapshot_is_available
)

print("SWeeP Genes logic is provided by src/pago_pipeline/sweep_genes_snapshot.py")
print("Vendored SWeeP package is loaded directly from scripts/sweep-2.1.3.0")

SWeeP Genes logic is provided by src/pago_pipeline/sweep_genes_snapshot.py
Vendored SWeeP package is loaded directly from scripts/sweep-2.1.3.0


In [7]:
# =============================================================================
# CELL 7 — Resolve active SWeeP Genes snapshot
# =============================================================================

sweep_genes_snapshot_payload = resolve_sweep_genes_snapshot(
    snapshot_mode=SWEEP_GENES_SNAPSHOT_MODE,
    snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
    source_fasta_snapshot_root_directory=FASTA_SNAPSHOT_ROOT_DIRECTORY,
    scripts_sweep_root_directory=SCRIPTS_SWEEP_ROOT_DIRECTORY,
    masks=SWEEP_GENES_MASKS,
    projected_dimensions_per_mask=PROJECTED_DIMENSIONS_PER_MASK,
    composition=COMPOSITION,
    projection=PROJECTION,
    random_seed=RANDOM_SEED,
    chunk_size=CHUNK_SIZE,
    n_jobs=N_JOBS,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

sweep_genes_snapshot_directory = sweep_genes_snapshot_payload["snapshot_directory"]
sweep_genes_manifest_file_path = sweep_genes_snapshot_payload["manifest_file_path"]
sweep_genes_manifest_payload = sweep_genes_snapshot_payload["manifest"]
sweep_genes_embeddings_file_path = sweep_genes_snapshot_payload["embeddings_file_path"]
sweep_genes_sequence_metadata_file_path = sweep_genes_snapshot_payload[
    "sequence_metadata_file_path"
]
sweep_genes_embeddings = sweep_genes_snapshot_payload["embeddings"]
sweep_genes_sequence_metadata_dataframe = sweep_genes_snapshot_payload[
    "sequence_metadata"
]
sweep_genes_output_directory = sweep_genes_snapshot_directory

print("Resolved SWeeP Genes snapshot successfully.")
print(f"Snapshot directory: {sweep_genes_snapshot_directory}")
print(f"Embeddings file path: {sweep_genes_embeddings_file_path}")
print(f"Sequence metadata file path: {sweep_genes_sequence_metadata_file_path}")

FileNotFoundError: Vendored sweep package root directory was not found: C:\pago-proj\pAgo-project\scripts\sweep-2.1.3.0.

In [8]:
# =============================================================================
# CELL 8 — Print SWeeP Genes snapshot summary
# =============================================================================

sweep_genes_embeddings_file_sha256 = sha256_of_file(
    input_file_path=sweep_genes_embeddings_file_path,
)
sweep_genes_manifest_file_sha256 = sha256_of_file(
    input_file_path=sweep_genes_manifest_file_path,
)

print("SWeeP Genes snapshot is ready.")
print(
    f"Snapshot created at UTC: {sweep_genes_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Sequence count: {sweep_genes_manifest_payload['sequence_count']}")
print(f"Embedding dimension: {sweep_genes_manifest_payload['embedding_dimension']}")
print(
    "Projected dimensions per mask: "
    f"{sweep_genes_manifest_payload['projected_dimensions_per_mask']}"
)
print(f"Masks used: {sweep_genes_manifest_payload['masks']}")
print(f"Chunk size: {sweep_genes_manifest_payload['chunk_size']}")
print(f"n_jobs: {sweep_genes_manifest_payload['n_jobs']}")
print(f"Embeddings file SHA-256: {sweep_genes_embeddings_file_sha256}")
print(f"Manifest file SHA-256: {sweep_genes_manifest_file_sha256}")

NameError: name 'sweep_genes_embeddings_file_path' is not defined

In [9]:
# =============================================================================
# CELL 9 — Preview sequence metadata and embedding statistics
# =============================================================================

preview_row_limit = 5
embedding_preview_dataframe = sweep_genes_sequence_metadata_dataframe.head(
    preview_row_limit
).copy()
embedding_preview_dataframe["embedding_l2_norm"] = np.linalg.norm(
    np.asarray(sweep_genes_embeddings[:preview_row_limit]),
    axis=1,
)

print(f"Embeddings array shape: {sweep_genes_embeddings.shape}")
print(f"Embeddings dtype: {sweep_genes_embeddings.dtype}")
print(
    "Source FASTA snapshot relative path: "
    f"{sweep_genes_manifest_payload['source_fasta_snapshot_relative_path']}"
)
print(
    "Source metadata snapshot relative path: "
    f"{sweep_genes_manifest_payload['source_metadata_snapshot_relative_path']}"
)
print(
    "Source XML snapshot relative path: "
    f"{sweep_genes_manifest_payload['source_xml_snapshot_relative_path']}"
)

embedding_preview_dataframe

NameError: name 'sweep_genes_sequence_metadata_dataframe' is not defined

In [ ]:
# =============================================================================
# CELL 10 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- protein_fasta_snapshot_directory")
print("- protein_fasta_file_path")
print("- protein_fasta_manifest_file_path")
print("- protein_fasta_manifest_payload")
print("- sweep_genes_snapshot_directory")
print("- sweep_genes_output_directory")
print("- sweep_genes_embeddings_file_path")
print("- sweep_genes_convenience_matrix_file_path")
print("- sweep_genes_manifest_file_path")
print("- sweep_genes_manifest_payload")
print("- sweep_genes_embeddings")
print("- sweep_genes_sequence_metadata_dataframe")